# 🧬 NESP - ThermoNet v2: Exploration & Visualisation des voxel features

Ce notebook fait 4 choses:
1. Import des librairies necessaires
2. Chargement des datasets (dataset.csv + features/*.npy)
3. Visualisation de la distribution de ddG, dT, et ddG vs dT (3 graphes)
4. Visualisation 3D des voxels pour 3 features: `occupancies`, `negative_ionizable`, `positive_ionizable` (wildtype vs mutant)

**Sources de donnees (Kaggle inputs a ajouter via "+ Add Input"):**
- Competition officielle: `novozymes-enzyme-stability-prediction`
- Notebook output: `vslaykovsky/14656-unique-mutations-voxel-features-pdbs`


## 1) Imports


In [15]:
import os
import glob

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from tqdm.notebook import tqdm

# IMPORTANT: sur Kaggle (et certaines versions recentes de plotly/Jupyter),
# 'init_notebook_mode' de l'ancienne API plotly.offline ne suffit plus pour
# afficher les graphes -> la cellule s'execute (Running -> Done) mais reste
# vide, sans aucune erreur. La solution est de fixer explicitement le
# renderer via plotly.io. 'iframe' fonctionne de maniere fiable sur Kaggle,
# meme sans connexion internet (il n'a pas besoin de charger le JS via CDN).
pio.renderers.default = 'iframe'

# Si jamais 'iframe' ne marche toujours pas dans ton environnement, essaie
# l'une de ces alternatives (decommente une ligne a la fois et re-teste) :
# pio.renderers.default = 'iframe_connected'
# pio.renderers.default = 'notebook_connected'
# pio.renderers.default = 'kaggle'

print('Renderer plotly actif:', pio.renderers.default)


Renderer plotly actif: iframe


### Test rapide: est-ce que plotly affiche bien quelque chose ?

Execute cette cellule seule. Si tu vois un petit graphe en barres, le renderer est bon -> continue normalement.
Si elle reste vide, essaie une des lignes alternatives dans la cellule d'imports ci-dessus (pio.renderers.default).


In [16]:
import plotly.express as _px_test
_test_fig = _px_test.bar(x=['a', 'b', 'c'], y=[1, 3, 2], title='Test renderer plotly')
_test_fig.show()


## 2) Chargement des datasets

**Important** : sur Kaggle, les datasets/notebooks ajoutes en input sont montes automatiquement sous `/kaggle/input/<nom>/`.

Avant d'executer ce notebook sur Kaggle :
1. Clique sur **+ Add Input** (panneau de droite)
2. Cherche et ajoute la competition **novozymes-enzyme-stability-prediction**
3. Cherche et ajoute le notebook **vslaykovsky/14656-unique-mutations-voxel-features-pdbs** (onglet "Notebook Output Files")

Les chemins ci-dessous correspondent a ces deux inputs.


In [17]:
# IMPORTANT: execute d'abord la cellule suivante (celle qui fait os.walk,
# uniquement sur Kaggle) pour voir la structure exacte, PUIS reviens
# corriger ces chemins si besoin.
#
# Detection automatique: local vs Kaggle
RUNNING_ON_KAGGLE = os.path.exists('/kaggle/input')
print('Environnement detecte:', 'Kaggle' if RUNNING_ON_KAGGLE else 'Local (PC)')

if RUNNING_ON_KAGGLE:
    # Nouvelle structure Kaggle (depuis leur changement recent):
    #   ../input/competitions/<competition-slug>/...
    #   ../input/datasets/<owner>/<dataset-slug>/...
    #   ../input/notebooks/<owner>/<notebook-slug>/...
    COMPETITION_DIR = '../input/competitions/novozymes-enzyme-stability-prediction'
    VOXEL_DATASET_DIR = '../input/notebooks/vslaykovsky/14656-unique-mutations-voxel-features-pdbs'
else:
    # MODE LOCAL (PC) : adapte ces 2 chemins vers l'endroit ou tu as telecharge
    # et decompresse les donnees sur ton disque.
    COMPETITION_DIR = './data/novozymes-enzyme-stability-prediction'
    VOXEL_DATASET_DIR = './data/14656-unique-mutations-voxel-features-pdbs'

WILDTYPE_PDB = os.path.join(COMPETITION_DIR, 'wildtype_structure_prediction_af2.pdb')
TEST_CSV = os.path.join(COMPETITION_DIR, 'test.csv')
CSV_PATH = os.path.join(VOXEL_DATASET_DIR, 'dataset.csv')
FEATURES_DIR = os.path.join(VOXEL_DATASET_DIR, 'features')

print('COMPETITION_DIR  :', COMPETITION_DIR)
print('VOXEL_DATASET_DIR:', VOXEL_DATASET_DIR)


Environnement detecte: Kaggle
COMPETITION_DIR  : ../input/competitions/novozymes-enzyme-stability-prediction
VOXEL_DATASET_DIR: ../input/notebooks/vslaykovsky/14656-unique-mutations-voxel-features-pdbs


In [18]:
# NOTE: Kaggle a change la structure de /kaggle/input/.
# Avant: /kaggle/input/<nom>/...
# Maintenant: /kaggle/input/{competitions,datasets,notebooks}/<owner>/<nom>/...
# On explore recursivement (jusqu'a 3 niveaux) pour trouver les vrais chemins.
# Cette cellule ne sert que si RUNNING_ON_KAGGLE=True (voir cellule suivante).
if RUNNING_ON_KAGGLE:
    print('Structure complete de ../input :')
    for root, dirs, files in os.walk('../input'):
        depth = root.count(os.sep) - '../input'.count(os.sep)
        if depth > 3:
            dirs[:] = []  # ne pas descendre plus bas
            continue
        indent = '  ' * depth
        print(f'{indent}{root}/')
        if depth == 3:
            for f in files[:5]:
                print(f'{indent}  - {f}')
            if len(files) > 5:
                print(f'{indent}  ... (+{len(files)-5} autres fichiers)')
else:
    print('Mode local detecte -> cette cellule est ignoree.')


Structure complete de ../input :
../input/
  ../input/datasets/
    ../input/datasets/vslaykovsky/
      ../input/datasets/vslaykovsky/thermonet-features/
        - nesp_features.npy
        - Q3214.npy
  ../input/competitions/
    ../input/competitions/novozymes-enzyme-stability-prediction/
  ../input/notebooks/
    ../input/notebooks/vslaykovsky/
      ../input/notebooks/vslaykovsky/14656-unique-mutations-voxel-features-pdbs/
        - __results__.html
        - __notebook__.ipynb
        - __output__.json
        - dataset.csv
        - custom.css
      ../input/notebooks/vslaykovsky/nesp-thermonet-v2/
        - ensemble_submission.csv
        - __results__.html
        - submission.csv
        - __notebook__.ipynb
        - __output__.json
        ... (+1 autres fichiers)


In [19]:
print(f'1. Chargement du fichier CSV: {CSV_PATH}')
df = pd.read_csv(CSV_PATH)
print(f'   Total mutations dans dataset.csv: {len(df)}')

# Construction du chemin vers le fichier .npy de chaque mutation
df['features_path'] = df.apply(
    lambda r: os.path.join(
        FEATURES_DIR, f'{r.PDB_chain}_{r.wildtype}{r.pdb_position}{r.mutant}.npy'
    ),
    axis=1,
)

# On garde seulement les mutations dont le fichier .npy existe reellement
df['features_exists'] = df['features_path'].apply(os.path.exists)
n_before = len(df)
df = df[df['features_exists']].reset_index(drop=True)
print(f'   Mutations avec features .npy disponibles: {len(df)} / {n_before}')

df.head()


1. Chargement du fichier CSV: ../input/notebooks/vslaykovsky/14656-unique-mutations-voxel-features-pdbs/dataset.csv
   Total mutations dans dataset.csv: 14656
   Mutations avec features .npy disponibles: 12167 / 14656


,sequence,wildtype,pdb_position,seq_position,mutant,ddG,dT,wT,pH,source,PDB_chain,features_path,features_exists
0,AAQASVVANQLIPINTALTLVMMRSEVVTPVGIPAEDIPRLVSMQV...,D,36,36,A,0.705833,NaN,NaN,7.0,"['Q3421.txt', 'Q3214_direct.csv', 'Q1744_direc...",1msiA,../input/notebooks/vslaykovsky/14656-unique-mu...,True
1,AAQASVVANQLIPINTALTLVMMRSEVVTPVGIPAEDIPRLVSMQV...,D,58,58,N,-0.120000,NaN,NaN,7.0,"['Q3421.txt', 'Q3214_direct.csv', 'Q1744_direc...",1msiA,../input/notebooks/vslaykovsky/14656-unique-mu...,True
2,AAQASVVANQLIPINTALTLVMMRSEVVTPVGIPAEDIPRLVSMQV...,E,25,25,A,-0.050000,NaN,NaN,7.0,"['Q3421.txt', 'Q3214_direct.csv', 'Q1744_direc...",1msiA,../input/notebooks/vslaykovsky/14656-unique-mu...,True
3,AAQASVVANQLIPINTALTLVMMRSEVVTPVGIPAEDIPRLVSMQV...,R,23,23,A,-0.763333,NaN,NaN,7.0,"['Q3421.txt', 'Q3214_direct.csv', 'Q1744_direc...",1msiA,../input/notebooks/vslaykovsky/14656-unique-mu...,True
4,AAQASVVANQLIPINTALTLVMMRSEVVTPVGIPAEDIPRLVSMQV...,R,39,39,A,-0.726667,NaN,NaN,7.0,"['Q3421.txt', 'Q3214_direct.csv', 'Q1744_direc...",1msiA,../input/notebooks/vslaykovsky/14656-unique-mu...,True


## 3) Visualisation de ddG, dT et ddG vs dT


In [20]:
# --- Graphe 1: distribution de ddG (Delta Delta G) ---
fig_ddg = px.histogram(
    df, x='ddG', nbins=60,
    title="Distribution de \u0394\u0394G (ddG) - changement d'energie de stabilite",
)
fig_ddg.update_layout(bargap=0.02)
fig_ddg.show()


In [21]:
# --- Graphe 2: distribution de dT (Delta T) ---
fig_dt = px.histogram(
    df, x='dT', nbins=60,
    title='Distribution de \u0394T (dT) - changement de temperature de fusion (Tm)',
)
fig_dt.update_layout(bargap=0.02)
fig_dt.show()


In [22]:
# --- Graphe 3: ddG vs dT (les deux targets sont normalement correles) ---
fig_scatter = px.scatter(
    df, x='ddG', y='dT',
    title='ddG vs dT',
    opacity=0.4,
    trendline='ols',  # ligne de regression pour voir la correlation
)
fig_scatter.show()


## 4) Visualisation 3D des voxels

Chaque fichier `.npy` a la forme `(14, 16, 16, 16)`:
- canaux 0-6  = 7 features du **wildtype**
- canaux 7-13 = 7 features du **mutant** (meme ordre de features)

Ordre des 7 features (standard ThermoNet / HTMD `getVoxelDescriptors`): `hydrophobic, aromatic, hbond_acceptor, hbond_donor, positive_ionizable, negative_ionizable, occupancies`


In [23]:
FEATURE_NAMES = [
    'hydrophobic',
    'aromatic',
    'hbond_acceptor',
    'hbond_donor',
    'positive_ionizable',
    'negative_ionizable',
    'occupancies',
]

VOXEL_SIZE = 16  # grille 16x16x16
GRID_COORDS = np.array(
    [(x, y, z) for x in range(VOXEL_SIZE) for y in range(VOXEL_SIZE) for z in range(VOXEL_SIZE)]
)


def plot_feature_diff(sample_idx: int, feature: str = 'occupancies',
                       threshold: float = 0.5, marker_size: int = 6):
    """
    Compare, voxel par voxel, wildtype vs mutant pour UNE feature choisie,
    et classe chaque voxel occupe (valeur > threshold) en 3 categories :

      - bleu  : occupe dans le wildtype ET le mutant (rien n'a change)
      - rouge : occupe SEULEMENT dans le mutant (nouvelle matiere ajoutee)
      - vert  : occupe SEULEMENT dans le wildtype (matiere disparue)

    threshold: peut/doit etre ajuste au cas par cas. Une occupation binaire
    (0 ou 1) marche bien avec 0.5, mais pour des features plus "floues"
    (hydrophobic, ionizable...) ou pour des mutations avec un ddG tres
    fort/faible, la distribution des valeurs peut etre differente -> essaie
    plusieurs valeurs (0.3, 0.5, 0.7...) et regarde laquelle donne la
    visualisation la plus lisible pour CE sample precis.
    """
    row = df.iloc[sample_idx]
    features = np.load(row.features_path)  # shape: (14, 16, 16, 16)

    ch = FEATURE_NAMES.index(feature)
    wildtype_vals = features[ch].flatten()
    mutant_vals = features[7 + ch].flatten()

    mask_wt = wildtype_vals > threshold
    mask_mut = mutant_vals > threshold

    both_mask = mask_wt & mask_mut          # present dans les 2 -> bleu
    mutant_only_mask = mask_mut & ~mask_wt  # apparu chez le mutant -> rouge
    wildtype_only_mask = mask_wt & ~mask_mut  # disparu chez le mutant -> vert

    categories = [
        (both_mask, 'blue', 'blue'),
        (mutant_only_mask, 'red', 'red'),
        (wildtype_only_mask, 'green', 'green'),
    ]

    fig = go.Figure()
    for mask, color, label in categories:
        fig.add_trace(
            go.Scatter3d(
                x=GRID_COORDS[mask, 0],
                y=GRID_COORDS[mask, 1],
                z=GRID_COORDS[mask, 2],
                mode='markers',
                marker=dict(size=marker_size, color=color, opacity=0.85),
                name=label,
                legendgroup='color',
            )
        )

    ddg_txt = f'{row.ddG:.2f}' if pd.notna(row.ddG) else 'NA'
    fig.update_layout(
        title=f'Train idx:{sample_idx}; ddg={ddg_txt}',
        height=750,
        legend_title_text='color',
        scene=dict(
            xaxis_title='x (voxel)',
            yaxis_title='y (voxel)',
            zaxis_title='z (voxel)',
        ),
    )
    # config explicite pour garantir l'interactivite (rotation/zoom a la souris)
    fig.show(config={'displayModeBar': True, 'scrollZoom': True})


In [24]:
# Exemple: visualiser les 3 premiers samples du dataset, feature "occupancies"
for i in range(min(3, len(df))):
    plot_feature_diff(i, feature='occupancies', threshold=0.5)

# Essaie aussi d'autres thresholds / features sur le meme sample pour comparer:
# plot_feature_diff(0, feature='occupancies', threshold=0.3)
# plot_feature_diff(0, feature='occupancies', threshold=0.7)
# plot_feature_diff(0, feature='hydrophobic', threshold=0.5)


## 5) Comparaison multi-features (occupancies / hydrophobic / hbond_donor)

Meme logique 3 couleurs (bleu = present dans les 2, rouge = mutant seulement,
vert = wildtype seulement), mais affichee **cote a cote** pour plusieurs
features en meme temps, sur un seul sample.


In [25]:
def plot_multi_feature_diff(sample_idx: int,
                             features=('occupancies', 'hydrophobic', 'hbond_donor'),
                             threshold: float = 0.5, marker_size: int = 5):
    """
    Meme principe que plot_feature_diff, mais affiche plusieurs features
    cote a cote (un subplot 3D par feature) pour comparer facilement.
    """
    row = df.iloc[sample_idx]
    arr = np.load(row.features_path)  # (14, 16, 16, 16)

    n = len(features)
    fig = make_subplots(
        rows=1, cols=n,
        specs=[[{'type': 'scatter3d'}] * n],
        subplot_titles=list(features),
    )

    colors = {'both': 'blue', 'mutant_only': 'red', 'wildtype_only': 'green'}

    for col, feat_name in enumerate(features, start=1):
        ch = FEATURE_NAMES.index(feat_name)
        wt_vals = arr[ch].flatten()
        mut_vals = arr[7 + ch].flatten()

        mask_wt = wt_vals > threshold
        mask_mut = mut_vals > threshold

        both_mask = mask_wt & mask_mut
        mutant_only_mask = mask_mut & ~mask_wt
        wildtype_only_mask = mask_wt & ~mask_mut

        for mask, key in [(both_mask, 'both'),
                           (mutant_only_mask, 'mutant_only'),
                           (wildtype_only_mask, 'wildtype_only')]:
            fig.add_trace(
                go.Scatter3d(
                    x=GRID_COORDS[mask, 0],
                    y=GRID_COORDS[mask, 1],
                    z=GRID_COORDS[mask, 2],
                    mode='markers',
                    marker=dict(size=marker_size, color=colors[key], opacity=0.85),
                    name=key,
                    legendgroup=key,
                    showlegend=(col == 1),  # une seule legende, pour la 1ere colonne
                ),
                row=1, col=col,
            )

    ddg_txt = f'{row.ddG:.2f}' if pd.notna(row.ddG) else 'NA'
    fig.update_layout(
        title=f'Train idx:{sample_idx}; ddg={ddg_txt} | threshold={threshold}',
        height=600,
        width=380 * n,
        legend_title_text='color',
    )
    fig.show(config={'displayModeBar': True, 'scrollZoom': True})


# Exemple
plot_multi_feature_diff(0, features=('occupancies', 'hydrophobic', 'hbond_donor'), threshold=0.5)


## 6) Statistiques : combien de voxels colores, et combien de samples disponibles ?

Deux questions distinctes :
- **Combien de voxels** (bleu/rouge/vert) apparaissent pour UN sample donne (sur un total de 16x16x16 = 4096 voxels par feature) ?
- **Combien de mutations (train idx)** avons-nous au total dans `df`, pretes a etre visualisees/entrainees (celles dont le fichier `.npy` existe reellement) ?


In [26]:
def count_diff_voxels(sample_idx: int, feature: str = 'occupancies', threshold: float = 0.5):
    """
    Retourne le nombre de voxels dans chaque categorie (both/mutant_only/
    wildtype_only) pour un sample et une feature donnes, sur un total de
    16*16*16 = 4096 voxels possibles.
    """
    row = df.iloc[sample_idx]
    arr = np.load(row.features_path)

    ch = FEATURE_NAMES.index(feature)
    wt_vals = arr[ch].flatten()
    mut_vals = arr[7 + ch].flatten()

    mask_wt = wt_vals > threshold
    mask_mut = mut_vals > threshold

    both = int((mask_wt & mask_mut).sum())
    mutant_only = int((mask_mut & ~mask_wt).sum())
    wildtype_only = int((mask_wt & ~mask_mut).sum())
    total_voxels = GRID_COORDS.shape[0]  # 4096

    return {
        'sample_idx': sample_idx,
        'feature': feature,
        'ddG': row.ddG,
        'both_blue': both,
        'mutant_only_red': mutant_only,
        'wildtype_only_green': wildtype_only,
        'total_colored': both + mutant_only + wildtype_only,
        'total_voxels': total_voxels,
    }


# --- 1) Nombre de voxels colores pour les 5 premiers samples ---
stats_rows = [count_diff_voxels(i, feature='occupancies', threshold=0.5)
              for i in range(min(5, len(df)))]
stats_df = pd.DataFrame(stats_rows)
print(stats_df)

print()
# --- 2) Nombre total de mutations (train idx) disponibles ---
print(f"Nombre total de mutations dans df (avec fichier .npy existant): {len(df)}")
print(f"Chaque sample a un espace de {GRID_COORDS.shape[0]} voxels possibles (16x16x16), par feature.")


   sample_idx      feature       ddG  both_blue  mutant_only_red  \
0           0  occupancies  0.705833       1545              127   
1           1  occupancies -0.120000        989               46   
2           2  occupancies -0.050000       1107               42   
3           3  occupancies -0.763333       1193               57   
4           4  occupancies -0.726667       1191               52   

   wildtype_only_green  total_colored  total_voxels  
0                  145           1817          4096  
1                   42           1077          4096  
2                   83           1232          4096  
3                  104           1354          4096  
4                  138           1381          4096  

Nombre total de mutations dans df (avec fichier .npy existant): 12167
Chaque sample a un espace de 4096 voxels possibles (16x16x16), par feature.


## 7) Comparaison de mutations avec des ddG differents

On cherche, pour chaque valeur de ddG demandee, la mutation la plus
proche dans `df` (recherche par distance minimale, car les valeurs
exactes en float peuvent varier legerement), puis on affiche les 3
mutations **cote a cote** avec le meme code couleur (bleu/rouge/vert)
pour voir comment la distribution des voxels occupes change selon
l'intensite (et le signe) du ddG.


In [27]:
def find_closest_idx(target_ddg: float):
    """Retourne l'index (dans df) de la mutation dont le ddG est le plus proche de target_ddg."""
    diffs = (df['ddG'] - target_ddg).abs()
    return diffs.idxmin()


def plot_ddg_comparison(ddg_values, feature: str = 'occupancies',
                         threshold: float = 0.5, marker_size: int = 5):
    """
    Affiche, cote a cote, la comparaison wildtype/mutant (bleu/rouge/vert)
    d'une feature donnee, pour plusieurs mutations choisies par leur ddG.
    """
    # 1) trouver l'idx le plus proche pour chaque ddG demande
    sample_indices = [find_closest_idx(v) for v in ddg_values]

    n = len(sample_indices)
    subplot_titles = []
    for target, idx in zip(ddg_values, sample_indices):
        actual_ddg = df.loc[idx, 'ddG']
        subplot_titles.append(f'idx={idx} | demande={target} | reel={actual_ddg:.4f}')

    fig = make_subplots(
        rows=1, cols=n,
        specs=[[{'type': 'scatter3d'}] * n],
        subplot_titles=subplot_titles,
    )

    colors = {'both': 'blue', 'mutant_only': 'red', 'wildtype_only': 'green'}
    ch = FEATURE_NAMES.index(feature)

    for col, idx in enumerate(sample_indices, start=1):
        row = df.loc[idx]
        arr = np.load(row.features_path)

        wt_vals = arr[ch].flatten()
        mut_vals = arr[7 + ch].flatten()

        mask_wt = wt_vals > threshold
        mask_mut = mut_vals > threshold

        both_mask = mask_wt & mask_mut
        mutant_only_mask = mask_mut & ~mask_wt
        wildtype_only_mask = mask_wt & ~mask_mut

        for mask, key in [(both_mask, 'both'),
                           (mutant_only_mask, 'mutant_only'),
                           (wildtype_only_mask, 'wildtype_only')]:
            fig.add_trace(
                go.Scatter3d(
                    x=GRID_COORDS[mask, 0],
                    y=GRID_COORDS[mask, 1],
                    z=GRID_COORDS[mask, 2],
                    mode='markers',
                    marker=dict(size=marker_size, color=colors[key], opacity=0.85),
                    name=key,
                    legendgroup=key,
                    showlegend=(col == 1),
                ),
                row=1, col=col,
            )

    fig.update_layout(
        title=f'Comparaison ddG (feature={feature}, threshold={threshold})',
        height=600,
        width=420 * n,
        legend_title_text='color',
    )
    fig.show(config={'displayModeBar': True, 'scrollZoom': True})

    # petit resume texte des comptes par categorie, pratique pour comparer les 3
    summary = []
    for target, idx in zip(ddg_values, sample_indices):
        summary.append(count_diff_voxels(idx, feature=feature, threshold=threshold))
    return pd.DataFrame(summary)


# --- Exemple avec les 3 valeurs de ddG demandees ---
ddg_targets = [0.705833, -0.120000, -0.050000]
summary_df = plot_ddg_comparison(ddg_targets, feature='occupancies', threshold=0.5)
print(summary_df)


   sample_idx      feature       ddG  both_blue  mutant_only_red  \
0           0  occupancies  0.705833       1545              127   
1           1  occupancies -0.120000        989               46   
2         791  occupancies -0.050000       1658               78   

   wildtype_only_green  total_colored  total_voxels  
0                  145           1817          4096  
1                   42           1077          4096  
2                   94           1830          4096  


## 8) Est-ce que ddG est vraiment correle avec la proportion de bleu ?

On teste l'hypothese sur un echantillon large (pas juste 3 exemples) :
on calcule, pour N mutations aleatoires, la fraction de voxels 'bleu'
(inchange) par rapport au total colore, et on regarde sa correlation
avec ddG. Si la correlation est faible/nulle, ca confirme que ddG n'est
PAS directement determine par la quantite de changement physique brut.


In [28]:
import numpy as np

N_SAMPLES = 500  # ajuste selon le temps de calcul disponible

sample_idx_list = np.random.RandomState(42).choice(len(df), size=min(N_SAMPLES, len(df)), replace=False)

rows = []
for idx in sample_idx_list:
    stats = count_diff_voxels(idx, feature='occupancies', threshold=0.5)
    total = stats['total_colored']
    blue_fraction = stats['both_blue'] / total if total > 0 else np.nan
    rows.append({'idx': idx, 'ddG': stats['ddG'], 'blue_fraction': blue_fraction,
                 'total_colored': total})

corr_df = pd.DataFrame(rows).dropna()

# Correlation de Pearson entre ddG et blue_fraction
correlation = corr_df['ddG'].corr(corr_df['blue_fraction'])
print(f"Correlation (Pearson) entre ddG et fraction de bleu: {correlation:.3f}")
print(f"(proche de 0 = pas de lien direct ; proche de +-1 = lien fort)")
print()
print(corr_df.describe())

# Visualisation
fig_corr = px.scatter(
    corr_df, x='ddG', y='blue_fraction',
    title=f'ddG vs fraction de voxels inchanges (bleu) | correlation={correlation:.3f}',
    opacity=0.5, trendline='ols',
)
fig_corr.show(config={'displayModeBar': True})


Correlation (Pearson) entre ddG et fraction de bleu: 0.072
(proche de 0 = pas de lien direct ; proche de +-1 = lien fort)

                idx         ddG  blue_fraction  total_colored
count    446.000000  446.000000     446.000000     446.000000
mean    5852.163677   -1.111162       0.801699    1672.147982
std     3551.203450    1.928548       0.146612     534.314674
min       19.000000  -10.300000       0.397590     265.000000
25%     2626.500000   -2.164485       0.764269    1266.500000
50%     5616.000000   -0.619167       0.856569    1644.500000
75%     9091.750000    0.027752       0.903154    2049.500000
max    12092.000000    7.930000       0.967769    3053.000000


**Split the Kfold**


In [29]:
from sklearn.model_selection import GroupKFold
import numpy as np

N_FOLDS = 5  # tu peux monter à 10 si tu as assez de compute

gkf = GroupKFold(n_splits=N_FOLDS)
df['fold'] = -1

# groupby sur PDB_chain : garantit que toutes les mutations d'une même
# structure PDB restent ensemble (soit toutes en train, soit toutes en val)
for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(df, groups=df['PDB_chain'])):
    df.loc[val_idx, 'fold'] = fold_idx

print(df['fold'].value_counts().sort_index())

fold
0    2434
1    2434
2    2433
3    2433
4    2433
Name: count, dtype: int64


**Dataset PyTorch (chargement + feature différentielle)**

In [30]:
import torch
from torch.utils.data import Dataset

class VoxelDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        arr = np.load(row.features_path).astype(np.float32)  # shape (14,16,16,16)

        # Feature différentielle : on remplace les 7 derniers canaux (mutant)
        # par la différence mutant - wildtype, comme dans ThermoNet2
        arr = arr.copy()
        arr[7:] -= arr[:7]

        x = torch.from_numpy(arr)  # (14, 16, 16, 16)
        y_ddg = torch.tensor(row.ddG, dtype=torch.float32)
        y_dt = torch.tensor(row.dT, dtype=torch.float32)

        return x, y_ddg, y_dt

**Architecture du modèle (ThermoNet2, deux têtes)**

In [31]:
import torch.nn as nn

class ThermoNet2(nn.Module):
    def __init__(self, in_channels=14, dropout=0.5):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv3d(in_channels, 16, kernel_size=3, padding=1),
            nn.SiLU(),
            nn.Conv3d(16, 16, kernel_size=3, padding=1),
            nn.SiLU(),
            nn.MaxPool3d(2),  # 16 -> 8

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.SiLU(),
            nn.Conv3d(32, 32, kernel_size=3, padding=1),
            nn.SiLU(),
            nn.MaxPool3d(2),  # 8 -> 4

            nn.Flatten(),
        )

        flat_dim = 32 * 4 * 4 * 4

        # tête ddG
        self.head_ddg = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(flat_dim, 64),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

        # tête dT
        self.head_dt = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(flat_dim, 64),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        features = self.backbone(x)
        ddg = self.head_ddg(features).squeeze(-1)
        dt = self.head_dt(features).squeeze(-1)
        return ddg, dt

**Bloc d'entraînement (une fonction par fold)**

In [32]:
from torch.utils.data import DataLoader

C_DT = 0.01  # poids du dT dans la loss multi-tâches, comme ThermoNet2

def train_one_fold(fold, df, epochs=20, batch_size=32, lr=1e-3, device='cuda'):
    train_df = df[df['fold'] != fold]
    val_df = df[df['fold'] == fold]

    train_loader = DataLoader(VoxelDataset(train_df), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(VoxelDataset(val_df), batch_size=batch_size, shuffle=False)

    model = ThermoNet2().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    mse = nn.MSELoss()

    losses = []
    ddg_losses = []
    dt_losses = []

    for epoch in range(epochs):
        model.train()
        epoch_loss, epoch_ddg_loss, epoch_dt_loss = 0.0, 0.0, 0.0

        for x, y_ddg, y_dt in train_loader:
            x, y_ddg, y_dt = x.to(device), y_ddg.to(device), y_dt.to(device)

            optimizer.zero_grad()
            pred_ddg, pred_dt = model(x)

            loss_ddg = mse(pred_ddg, y_ddg)
            loss_dt = mse(pred_dt, y_dt)
            loss = loss_ddg + C_DT * loss_dt

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * x.size(0)
            epoch_ddg_loss += loss_ddg.item() * x.size(0)
            epoch_dt_loss += loss_dt.item() * x.size(0)

        n = len(train_df)
        losses.append(epoch_loss / n)
        ddg_losses.append(epoch_ddg_loss / n)
        dt_losses.append(epoch_dt_loss / n)

        print(f"[Fold {fold}] Epoch {epoch+1}/{epochs} - loss={losses[-1]:.4f} "
              f"(ddg={ddg_losses[-1]:.4f}, dt={dt_losses[-1]:.4f})")

    return model, val_loader, {'losses': losses, 'ddg_losses': ddg_losses, 'dt_losses': dt_losses}

**Boucle complète sur tous les folds**

In [33]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

all_results = []

for fold in range(N_FOLDS):
    model, val_loader, train_history = train_one_fold(fold, df, epochs=20, device=DEVICE)
    eval_result = evaluate_fold(model, val_loader, device=DEVICE)
    all_results.append({**train_history, **eval_result})

mean_spearman_ddg = np.mean([r['corr_ddg'] for r in all_results])
print(f"\nSpearman ddG moyen sur {N_FOLDS} folds: {mean_spearman_ddg:.4f}")

Device: cuda


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning:


    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning:


    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning:


Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/




AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
